# 01 · Feature Distributions

**Purpose.** Распределения 26 whitelist-фич, поиск мёртвых/константных/дублирующих признаков и корреляционная структура.

**What to look for.**
- dead/constant фичи (std~0) и near-dead (zero_rate~1)
- m1_flag_end_of_period — действительно ли всегда 0
- m1_signal vs m1_signal_final — идентичны ли
- сильно скоррелированные пары (возможные дубли)

> Это lab-ноутбук: выводы здесь предварительные, не финальный отчёт. Меняй параметры в ячейке *Parameters* и перезапускай.

In [ ]:
# --- bootstrap: запуск из корня проекта (рядом с data/ и backend/) ---
import sys, os
from pathlib import Path
# найти корень проекта и встать в него
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'processed').is_dir()), _here)
os.chdir(_root)
sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)
from lab import utils as u
print('project root:', _root.name)

In [ ]:
d = u.load_final_dataset()
av = u.available_whitelist(d)
len(av)

### Сводка по всем whitelist-фичам

In [ ]:
u.summarize_features(d, av)

### Поиск dead / constant / near-dead

In [ ]:
u.find_dead_or_constant(d, av)

### m1_flag_end_of_period — частота единиц

In [ ]:
s = pd.to_numeric(d['m1_flag_end_of_period'], errors='coerce')
print('value counts:'); print(s.value_counts(dropna=False))
print('share == 1:', round(float((s==1).mean()),4))

### m1_signal vs m1_signal_final — идентичность

In [ ]:
a = pd.to_numeric(d['m1_signal'], errors='coerce')
b = pd.to_numeric(d['m1_signal_final'], errors='coerce')
print('max abs diff:', float((a-b).abs().max()))
print('identical rows share:', round(float(((a-b).abs()<1e-12).mean()),4))
print('corr:', round(u.pearson(a,b),6))

### Корреляционная матрица whitelist (Spearman)

In [ ]:
fig, ax, corr = u.correlation_heatmap(d, av, method='spearman'); plt.show()

### Сильно скоррелированные пары (|rho| > порога)

## Parameters
Меняй значения здесь и перезапускай ноутбук.

In [ ]:
CORR_THRESHOLD = 0.9

In [ ]:
import itertools
pairs = []
for i,j in itertools.combinations(av,2):
    r = u.spearman(d[i], d[j])
    if abs(r) >= CORR_THRESHOLD:
        pairs.append({'a':i,'b':j,'spearman':round(r,3)})
pd.DataFrame(pairs).sort_values('spearman', key=lambda s: s.abs(), ascending=False)

### Распределения по модулям M1–M5
Меняй модуль в параметре.

## Parameters
Меняй значения здесь и перезапускай ноутбук.

In [ ]:
MODULE = 'm1'   # m1 / m2 / m3 / m4 / m5

In [ ]:
mod_feats = u.split_features_by_module(av).get(MODULE, [])
for f in mod_feats:
    fig, axes = u.plot_feature_distribution(d, f); plt.show()

### M4 Tax_Week_Flag частота и M5 Roskazna до/после 2021

In [ ]:
print('Tax_Week_Flag share==1:', round(float((pd.to_numeric(d['m4_Tax_Week_Flag'],errors='coerce')==1).mean()),3))
rk = 'm5_roskazna_net_flow_stress_mad_score'
if rk in d.columns:
    pre = d[d['date'] < '2021-01-01'][rk]
    post = d[d['date'] >= '2021-01-01'][rk]
    print(f'{rk}: pre-2021 nonzero share', round(float((pre.fillna(0)!=0).mean()),3),
          '| post-2021', round(float((post.fillna(0)!=0).mean()),3))

## Notes / Open questions

- Подтверди: m1_signal_final == m1_signal (дубль) и flag_end_of_period вклад.
- Какие ещё пары почти дублируют друг друга — кандидаты на чистку whitelist.
- M2/M3 sparsity — учитывать при интерпретации вкладов модулей.